**File:** eval/error_analysis.ipynb  
**Owner:** Shamathmika

**Purpose:**  
Identifies DocRes failure cases by finding the 10 worst-performing images  
by PSNR, displays degraded → restored → clean for each, categorizes failure  
types, and summarizes the model's blind spots.

**Dependencies:**  
- `eval/outputs/results_docres.csv` — produced by `eval/run_eval.py`  
- CSV must have columns: `degraded_path`, `restored_path`, `clean_path`, `psnr`, `ssim`, `cer`

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DocRestore'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

RESULTS_CSV = Path('eval/outputs/results_docres.csv')
N_WORST = 10

if not RESULTS_CSV.exists():
    raise FileNotFoundError(
        f'{RESULTS_CSV} not found. Run eval/run_eval.py first.'
    )

df = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(df)} evaluation results')
print(f'Columns: {list(df.columns)}')
df.describe()

In [ ]:
# Find the 10 worst-performing images by PSNR (lowest = worst)
worst = df.nsmallest(N_WORST, 'psnr').reset_index(drop=True)
print(f'Top {N_WORST} worst by PSNR:')
print(worst[['degraded_path', 'psnr', 'ssim', 'cer']].to_string(index=True))

In [ ]:
# Display degraded -> restored -> clean for each failure case
def load_thumb(path, size=(200, 200)):
    return Image.open(path).convert('RGB').resize(size, Image.LANCZOS)

fig, axes = plt.subplots(N_WORST, 3, figsize=(9, N_WORST * 3))
fig.suptitle('Failure Cases — Degraded | Restored | Clean', fontsize=13, fontweight='bold')

col_labels = ['Degraded', 'Restored', 'Clean']
for col, label in enumerate(col_labels):
    axes[0, col].set_title(label, fontsize=10, fontweight='bold')

for i, row in worst.iterrows():
    images = [
        load_thumb(row['degraded_path']),
        load_thumb(row['restored_path']),
        load_thumb(row['clean_path']),
    ]
    for col, img in enumerate(images):
        axes[i, col].imshow(img)
        axes[i, col].axis('off')

    axes[i, 0].set_ylabel(
        f'#{i+1}  PSNR={row["psnr"]:.2f}  SSIM={row["ssim"]:.3f}  CER={row["cer"]:.3f}',
        fontsize=7, rotation=0, labelpad=120, va='center'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Categorize failure types by inspecting degraded image filename patterns
# Augraphy stage names embedded in filenames (if present) or manual inspection
import collections

keywords = {
    'fold':       'Folding',
    'bleed':      'Ink bleed / bleed-through',
    'drum':       'Dirty drum / stains',
    'noise':      'Noise',
    'jpeg':       'JPEG artifacts',
    'markup':     'Markup',
    'brightness': 'Low brightness / contrast',
}

category_counts = collections.Counter()
for _, row in worst.iterrows():
    stem = Path(row['degraded_path']).stem.lower()
    matched = False
    for kw, label in keywords.items():
        if kw in stem:
            category_counts[label] += 1
            matched = True
    if not matched:
        category_counts['Unknown / mixed'] += 1

print('Failure type distribution among worst 10:')
for cat, count in category_counts.most_common():
    print(f'  {cat}: {count}')

## TODO: Summary

**Blind spots**  
- [ ] Which degradation types does DocRes struggle with most?
- [ ] Are failures concentrated in specific image regions (edges, dense text)?

**Metrics analysis**  
- [ ] Is low PSNR correlated with high CER? (i.e. do visual failures hurt OCR?)
- [ ] Are SSIM and PSNR consistent, or do they disagree on failure cases?

**Potential fixes**  
- [ ] What augmentation or architectural changes could address the worst failure types?